In [1]:
pip install numpy pandas matplotlib seaborn scikit-learn openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
ac = pd.read_excel("active_customer.xlsx")
statements = pd.read_excel("capexmonthend_accounts_cc.xlsx")
transactions = pd.read_excel("stg_aa_ctransaction_cc_rec_mdl.xlsx")

In [4]:
# EDA and data cleaning steps

In [5]:
ac.head()

,customer_account_no,last_billing_date,status
0,cust_111,2025-03-28,Active
1,cust_111,2025-04-28,Active
2,cust_111,2025-05-28,Active
3,cust_112,2025-03-06,Active
4,cust_112,2025-04-06,Active


In [6]:
statements.head()

,customer_account_no,last_billing_date,current_overdue_cycles,minimum_amount,min_amt_due_date,stmt_next_date,number_of_days_in_arreas
0,cust_111,2025-03-28,0,-150.0,2025-04-21,2025-04-28,0
1,cust_111,2025-04-28,0,-200.0,2025-05-21,2025-05-28,0
2,cust_111,2025-05-28,0,-100.0,2025-06-21,2025-06-28,0
3,cust_112,2025-03-06,0,-100.0,2025-03-26,2025-04-06,0
4,cust_112,2025-04-06,0,-100.0,2025-04-26,2025-05-06,0


In [7]:
transactions.head()

,customer_account_no,amount,post_date,tansaction_remark
0,cust_123,300.00,2025-03-28,PAYMENT RECEIVED - TRANSFER
1,cust_123,100.00,2025-04-16,PAYMENT RECEIVED - TRANSFER
2,cust_123,4589.36,2025-04-28,ANNUAL FEE
3,cust_123,300.00,2025-05-20,PAYMENT RECEIVED - MOBILE
4,cust_121,4000.00,2025-03-29,PAYMENT RECEIVED - TRANSFER


In [8]:
# check null values of each dataframe
print("Null values in active customers:")
print(ac.isnull().sum())    
print("\nNull values in statements:")
print(statements.isnull().sum())
print("\nNull values in transactions:")
print(transactions.isnull().sum())

Null values in active customers:
customer_account_no    0
last_billing_date      0
status                 0
dtype: int64

Null values in statements:
customer_account_no         0
last_billing_date           0
current_overdue_cycles      0
minimum_amount              0
min_amt_due_date            0
stmt_next_date              0
number_of_days_in_arreas    0
dtype: int64

Null values in transactions:
customer_account_no     0
amount                  0
post_date               0
tansaction_remark      92
dtype: int64


In [9]:
# check duplicate values in each dataframe
print("Duplicate values in active customers:", ac.duplicated().sum())
print("Duplicate values in statements:", statements.duplicated().sum())
print("Duplicate values in transactions:", transactions.duplicated().sum())

Duplicate values in active customers: 0
Duplicate values in statements: 0
Duplicate values in transactions: 12


In [10]:
# show the duplicates in transactions
duplicate_transactions = transactions[transactions.duplicated()]
print("Duplicate transactions:")
print(duplicate_transactions)

Duplicate transactions:
   customer_account_no  amount  post_date tansaction_remark
26            cust_122    10.0 2025-04-01               NaN
27            cust_122    10.0 2025-04-01               NaN
45            cust_124    33.2 2025-03-02               NaN
65            cust_125     6.5 2025-03-09               NaN
66            cust_125     6.5 2025-03-09               NaN
78            cust_125   475.0 2025-04-21               NaN
83            cust_125  4000.0 2025-04-22               NaN
84            cust_125  4000.0 2025-04-22               NaN
85            cust_125  4000.0 2025-04-22               NaN
86            cust_125  4000.0 2025-04-22               NaN
87            cust_125  4000.0 2025-04-22               NaN
89            cust_125  4000.0 2025-04-23               NaN


In [11]:
# drop duplicate values since transaction remarks are null here and we don't know that the payment has recieved twice or multiple times or trasaction never received and the record is duplicated due to some error in data collection
transactions = transactions.drop_duplicates()

In [13]:
ac.info()

<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   customer_account_no  48 non-null     str           
 1   last_billing_date    48 non-null     datetime64[us]
 2   status               48 non-null     str           
dtypes: datetime64[us](1), str(2)
memory usage: 1.3 KB


In [14]:
statements.info()

<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   customer_account_no       48 non-null     str           
 1   last_billing_date         48 non-null     datetime64[us]
 2   current_overdue_cycles    48 non-null     int64         
 3   minimum_amount            48 non-null     float64       
 4   min_amt_due_date          48 non-null     datetime64[us]
 5   stmt_next_date            48 non-null     datetime64[us]
 6   number_of_days_in_arreas  48 non-null     int64         
dtypes: datetime64[us](3), float64(1), int64(2), str(1)
memory usage: 2.8 KB


In [16]:
transactions.info()

<class 'pandas.DataFrame'>
Index: 150 entries, 0 to 161
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   customer_account_no  150 non-null    str           
 1   amount               150 non-null    float64       
 2   post_date            150 non-null    datetime64[us]
 3   tansaction_remark    70 non-null     str           
dtypes: datetime64[us](1), float64(1), str(2)
memory usage: 5.9 KB


In [17]:
# filter active customers
active_customers = ac[ac['status'] == 'Active']
print(f"Number of active customers: {len(active_customers)}")

Number of active customers: 46


In [18]:
# filter transaction remarks starting with "PAYMENT"
paid_transactions = transactions[transactions['tansaction_remark'].str.upper().str.startswith('PAYMENT')]
print(f"Number of payment transactions: {len(paid_transactions)}")

Number of payment transactions: 57


In [19]:
# 1. Enforce Datatype Consistency (Crucial for date logic)
statements['last_billing_date'] = pd.to_datetime(statements['last_billing_date'])
statements['min_amt_due_date'] = pd.to_datetime(statements['min_amt_due_date'])
paid_transactions['post_date'] = pd.to_datetime(paid_transactions['post_date'])

# Ensure amounts are absolute values (in case the bank logs payments as negatives)
paid_transactions['amount'] = paid_transactions['amount'].abs()
statements['minimum_amount'] = statements['minimum_amount'].abs()



In [20]:
# Inner join statements with the active customers list. 
# This instantly drops any statements belonging to inactive/closed accounts.
active_statements = pd.merge(
    statements, 
    ac[['customer_account_no']], 
    on='customer_account_no', 
    how='inner'
)

# Left Join Payments to the Active Statements
merged_df = pd.merge(
    active_statements, 
    paid_transactions, 
    on='customer_account_no', 
    how='left'
)

# Apply the Time Boundary (The Window Rule)
valid_window_mask = (
    merged_df['post_date'].notna() & 
    (merged_df['post_date'] >= merged_df['last_billing_date']) & 
    (merged_df['post_date'] <= merged_df['min_amt_due_date'])
)

# Isolate Valid Payments (Zero out invalid/late payments)
merged_df['valid_payment_amount'] = np.where(valid_window_mask, merged_df['amount'], 0)

# Aggregate Back to the Statement Level
final_cycle_payments = merged_df.groupby(
    ['customer_account_no', 'last_billing_date', 'min_amt_due_date', 'minimum_amount']
)['valid_payment_amount'].sum().reset_index()

final_cycle_payments.rename(columns={'valid_payment_amount': 'total_paid'}, inplace=True)

final_cycle_payments.head()

,customer_account_no,last_billing_date,min_amt_due_date,minimum_amount,total_paid
0,cust_111,2025-03-28,2025-04-21,150.0,300.0
1,cust_111,2025-04-28,2025-05-21,200.0,19956.0
2,cust_111,2025-05-28,2025-06-21,100.0,12000.0
3,cust_112,2025-03-06,2025-03-26,100.0,12000.0
4,cust_112,2025-04-06,2025-04-26,100.0,0.0


In [21]:
# Apply the Bad Flag Logic
final_cycle_payments['is_bad'] = np.where(
    final_cycle_payments['total_paid'] < final_cycle_payments['minimum_amount'], 
    1, 
    0
)

# Calculate the Final Cycle-Wise Bad Rate
# Extract the billing cycle day (6th, 8th, 12th, 28th)
final_cycle_payments['billing_cycle_day'] = final_cycle_payments['last_billing_date'].dt.day

# Group by the cycle day to get our final KPIs
bad_rate_summary = final_cycle_payments.groupby('billing_cycle_day').agg(
    total_active_statements=('customer_account_no', 'count'),
    bad_statements=('is_bad', 'sum')
).reset_index()

# Calculate the actual rate
bad_rate_summary['bad_rate_percentage'] = (bad_rate_summary['bad_statements'] / bad_rate_summary['total_active_statements']) * 100

print(bad_rate_summary)

   billing_cycle_day  total_active_statements  bad_statements  \
0                  6                       12               7   
1                  8                       12               2   
2                 12                       15               0   
3                 28                        9               0   

   bad_rate_percentage  
0            58.333333  
1            16.666667  
2             0.000000  
3             0.000000  


## ETL Pipeline
The above steps successfully implement the logical requirements. We can now wrap this logic into an object-oriented ETL (Extract, Transform, Load) pipeline for future reusability.

In [3]:
import pandas as pd
import numpy as np

class PDMPipeline:
    def __init__(self, active_cust_path, statement_path, transaction_path):
        self.active_cust_path = active_cust_path
        self.statement_path = statement_path
        self.transaction_path = transaction_path

    def extract(self):
        print("Extracting data...")
        self.ac = pd.read_excel(self.active_cust_path)
        self.statements = pd.read_excel(self.statement_path)
        self.transactions = pd.read_excel(self.transaction_path)

    def transform(self):
        print("Transforming data...")
        # 1. Clean Data
        active_customers = self.ac[self.ac['status'] == 'Active'].copy()
        
        paid_txns = self.transactions.dropna(subset=['tansaction_remark']).copy()
        paid_txns = paid_txns[paid_txns['tansaction_remark'].str.upper().str.startswith('PAYMENT')]
        paid_txns = paid_txns.drop_duplicates()
        
        self.statements['last_billing_date'] = pd.to_datetime(self.statements['last_billing_date'])
        self.statements['min_amt_due_date'] = pd.to_datetime(self.statements['min_amt_due_date'])
        paid_txns['post_date'] = pd.to_datetime(paid_txns['post_date'])
        
        self.statements['minimum_amount'] = self.statements['minimum_amount'].abs()
        paid_txns['amount'] = paid_txns['amount'].abs()
        
        # 2. Join Data
        active_stmts = pd.merge(self.statements, active_customers[['customer_account_no']], on='customer_account_no', how='inner')
        merged_df = pd.merge(active_stmts, paid_txns, on='customer_account_no', how='left')
        
        # 3. Time Boundary Logic
        valid_window = (
            merged_df['post_date'].notna() &
            (merged_df['post_date'] >= merged_df['last_billing_date']) &
            (merged_df['post_date'] <= merged_df['min_amt_due_date'])
        )
        merged_df['valid_payment_amount'] = np.where(valid_window, merged_df['amount'], 0)
        
        # 4. Aggregation
        final_cycle_payments = merged_df.groupby(
            ['customer_account_no', 'last_billing_date', 'min_amt_due_date', 'minimum_amount']
        )['valid_payment_amount'].sum().reset_index()
        final_cycle_payments.rename(columns={'valid_payment_amount': 'total_paid'}, inplace=True)
        
        # 5. Bad Rate Calculation
        final_cycle_payments['is_bad'] = np.where(
            final_cycle_payments['total_paid'] < final_cycle_payments['minimum_amount'], 1, 0
        )
        final_cycle_payments['billing_cycle_day'] = final_cycle_payments['last_billing_date'].dt.day
        
        bad_rate_summary = final_cycle_payments.groupby('billing_cycle_day').agg(
            total_active_statements=('customer_account_no', 'count'),
            bad_statements=('is_bad', 'sum')
        ).reset_index()
        bad_rate_summary['bad_rate_percentage'] = (bad_rate_summary['bad_statements'] / bad_rate_summary['total_active_statements']) * 100
        
        self.bad_rate_summary = bad_rate_summary
        return self.bad_rate_summary

    def load(self, output_path="pdm_bad_rate_summary.csv"):
        print(f"Loading data into {output_path}...")
        self.bad_rate_summary.to_csv(output_path, index=False)
        print("Pipeline completed successfully!")

    def run(self):
        self.extract()
        self.transform()
        self.load()
        return self.bad_rate_summary


In [4]:
# Run the pipeline
pipeline = PDMPipeline(
    active_cust_path="active_customer.xlsx",
    statement_path="capexmonthend_accounts_cc.xlsx",
    transaction_path="stg_aa_ctransaction_cc_rec_mdl.xlsx"
)
summary_df = pipeline.run()
summary_df

Extracting data...
Transforming data...
Loading data into pdm_bad_rate_summary.csv...
Pipeline completed successfully!


,billing_cycle_day,total_active_statements,bad_statements,bad_rate_percentage
0,6,12,7,58.333333
1,8,12,2,16.666667
2,12,15,0,0.000000
3,28,9,0,0.000000
